# Archived Spark implementation
Historical source from the previous laptop. Run the notebooks one directory above for the current portable pipeline. Outputs have been cleared to avoid presenting old results as current.

# 02 - Join, segment, and build the integrated recommendation graph

This notebook joins the independently cleaned Spotify track/artist catalog from notebook 01 with the independently cleaned SPUD playlist data from notebook 01b. It creates the heterogeneous graph required by the dissertation:

- playlist nodes;
- track nodes;
- artist nodes;
- playlist-to-track target edges;
- track-to-artist context edges.

All playlist tracks are retained even when the original 600k-track catalog has no matching audio row. This is essential because deleting unmatched tracks would remove most of the real playlist graph. No train/test split or model is created here.

## 1. Environment and verified inputs

Run `01_preprocessing.ipynb` and `01b_playlist_preprocessing.ipynb` before this notebook.

In [ ]:
import os
import sys
import json
import math
from pathlib import Path


def find_project_root():
    required_paths = [
        Path("data/processed/tracks_enriched.parquet"),
        Path("data/processed/track_artist_joined.parquet"),
        Path("data/processed/playlists/spud_playlists_clean.parquet"),
        Path("data/processed/playlists/spud_tracks_clean.parquet"),
        Path("data/processed/playlists/spud_playlist_track_edges_clean.parquet"),
    ]
    for anchor in [Path.cwd().resolve(), Path(sys.executable).resolve()]:
        for parent in [anchor, *anchor.parents]:
            for candidate in [parent, parent / "art_xharra"]:
                if all((candidate / path).exists() for path in required_paths):
                    return candidate.resolve()
    raise FileNotFoundError(
        "Required processed inputs were not found. Run notebooks 01 and 01b first."
    )


PROJECT_ROOT = find_project_root()
JDK_ROOT = PROJECT_ROOT / ".tools/jdk17/jdk-17.0.20+8"
HADOOP_ROOT = PROJECT_ROOT / ".tools/hadoop"
if not JDK_ROOT.exists():
    raise FileNotFoundError(f"Project Java runtime is missing: {JDK_ROOT}")
os.environ["JAVA_HOME"] = str(JDK_ROOT)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = str(JDK_ROOT / "bin") + os.pathsep + os.environ.get("PATH", "")
if os.name == "nt":
    if not (HADOOP_ROOT / "bin/winutils.exe").exists():
        raise FileNotFoundError(f"Windows Hadoop helper is missing: {HADOOP_ROOT / 'bin/winutils.exe'}")
    os.environ["HADOOP_HOME"] = str(HADOOP_ROOT)
    os.environ["PATH"] = str(HADOOP_ROOT / "bin") + os.pathsep + os.environ["PATH"]

from pyspark import StorageLevel
from pyspark.ml.feature import HashingTF, RegexTokenizer
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession, Window, functions as F
import pandas as pd

spark = (
    SparkSession.builder.master("local[*]")
    .appName("Integrated-Playlist-Track-Artist-Graph")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

TRACK_CATALOG_PATH = PROJECT_ROOT / "data/processed/tracks_enriched.parquet"
ARTIST_CATALOG_PATH = PROJECT_ROOT / "data/processed/track_artist_joined.parquet"
PLAYLISTS_PATH = PROJECT_ROOT / "data/processed/playlists/spud_playlists_clean.parquet"
SPUD_TRACKS_PATH = PROJECT_ROOT / "data/processed/playlists/spud_tracks_clean.parquet"
PLAYLIST_EDGES_PATH = PROJECT_ROOT / "data/processed/playlists/spud_playlist_track_edges_clean.parquet"
OUTPUT_ROOT = PROJECT_ROOT / "data/processed/gnn_integrated"
PLAYLIST_NODES_PATH = OUTPUT_ROOT / "playlist_nodes.parquet"
TRACK_NODES_PATH = OUTPUT_ROOT / "track_nodes.parquet"
ARTIST_NODES_PATH = OUTPUT_ROOT / "artist_nodes.parquet"
PLAYLIST_TRACK_EDGES_PATH = OUTPUT_ROOT / "playlist_track_edges.parquet"
TRACK_ARTIST_EDGES_PATH = OUTPUT_ROOT / "track_artist_edges.parquet"
FEATURE_CATALOG_PATH = OUTPUT_ROOT / "feature_catalog.parquet"
REPORTS_ROOT = PROJECT_ROOT / "reports"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Spark       : {spark.version}")

## 2. Load all cleaned datasets and print their types

In [ ]:
playlist_dimension = spark.read.parquet(str(PLAYLISTS_PATH)).persist(StorageLevel.MEMORY_AND_DISK)
spud_track_dimension = spark.read.parquet(str(SPUD_TRACKS_PATH)).persist(StorageLevel.MEMORY_AND_DISK)
playlist_edge_source = spark.read.parquet(str(PLAYLIST_EDGES_PATH)).persist(StorageLevel.MEMORY_AND_DISK)
catalog_tracks = spark.read.parquet(str(TRACK_CATALOG_PATH)).persist(StorageLevel.MEMORY_AND_DISK)
catalog_artist_relationships = spark.read.parquet(str(ARTIST_CATALOG_PATH)).persist(StorageLevel.MEMORY_AND_DISK)

input_counts = {
    "spud_playlists": playlist_dimension.count(),
    "spud_tracks": spud_track_dimension.count(),
    "spud_playlist_track_edges": playlist_edge_source.count(),
    "catalog_tracks": catalog_tracks.count(),
    "catalog_artist_relationships": catalog_artist_relationships.count(),
}

def schema_as_pandas(dataset_name, dataframe):
    return pd.DataFrame([
        {"dataset": dataset_name, "column": field.name,
         "spark_type": field.dataType.simpleString(), "nullable": field.nullable}
        for field in dataframe.schema.fields
    ])

input_schema_pdf = pd.concat([
    schema_as_pandas("spud_playlists", playlist_dimension),
    schema_as_pandas("spud_tracks", spud_track_dimension),
    schema_as_pandas("spud_playlist_track_edges", playlist_edge_source),
    schema_as_pandas("catalog_tracks", catalog_tracks),
    schema_as_pandas("catalog_artist_relationships", catalog_artist_relationships),
], ignore_index=True)
display(input_schema_pdf)
display(pd.DataFrame(input_counts.items(), columns=["dataset", "rows"]))

## 3. Join playlist tracks to the original audio catalog

The graph universe consists only of tracks that occur in at least one playlist. The join is left-sided from those tracks because audio-feature coverage is limited. `audio_features_available` records whether a match was found.

In [ ]:
connected_track_ids = playlist_edge_source.select("spotify_track_id").distinct()
connected_spud_tracks = (
    connected_track_ids.join(spud_track_dimension, on="spotify_track_id", how="inner")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
connected_track_count = connected_spud_tracks.count()
connected_playlist_count = playlist_edge_source.select("playlist_id").distinct().count()

audio_catalog = catalog_tracks.select(
    F.col("id").alias("spotify_track_id"),
    F.col("name").alias("catalog_track_name"),
    F.col("popularity").alias("catalog_popularity_context"),
    F.col("explicit").alias("catalog_explicit"),
    F.col("danceability").alias("catalog_danceability"),
    F.col("energy").alias("catalog_energy"),
    F.col("key").alias("catalog_key"),
    F.col("loudness").alias("catalog_loudness"),
    F.col("mode").alias("catalog_mode"),
    F.col("speechiness").alias("catalog_speechiness"),
    F.col("acousticness").alias("catalog_acousticness"),
    F.col("instrumentalness").alias("catalog_instrumentalness"),
    F.col("liveness").alias("catalog_liveness"),
    F.col("valence").alias("catalog_valence"),
    F.col("tempo").alias("catalog_tempo"),
    F.col("time_signature").alias("catalog_time_signature"),
    F.col("release_year").alias("catalog_release_year"),
    F.col("tempo_was_missing").alias("catalog_tempo_was_missing"),
    F.lit(1).alias("_audio_match"),
)
joined_tracks = (
    connected_spud_tracks.join(audio_catalog, on="spotify_track_id", how="left")
    .withColumn("audio_features_available", F.col("_audio_match").isNotNull().cast("int"))
    .drop("_audio_match")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
audio_match_count = joined_tracks.where("audio_features_available = 1").count()
audio_match_pct = 100 * audio_match_count / connected_track_count

playlist_audio_coverage = (
    playlist_edge_source.join(
        joined_tracks.select("spotify_track_id", "audio_features_available"),
        on="spotify_track_id", how="inner",
    )
    .groupBy("playlist_id").agg(
        F.count("spotify_track_id").alias("playlist_track_count"),
        F.sum("audio_features_available").alias("audio_matched_track_count"),
    )
    .withColumn("audio_coverage_ratio", F.col("audio_matched_track_count") / F.col("playlist_track_count"))
)
print(f"Connected playlist tracks : {connected_track_count:,}")
print(f"Tracks with audio features: {audio_match_count:,} ({audio_match_pct:.3f}%)")
print(f"Connected playlists      : {connected_playlist_count:,}")
playlist_audio_coverage.select("audio_coverage_ratio").summary("min", "25%", "50%", "75%", "max").show()

## 4. Segment and encode track nodes

All tracks receive SPUD duration, title-hash, and availability features. Audio values are standardized using only matched catalog tracks; unmatched tracks receive zero (the standardized mean) plus `audio_features_available = 0`. Popularity is isolated in an optional context vector and excluded from `features_safe`.

In [ ]:
TRACK_TITLE_HASH_DIM = 32
AUDIO_CONTINUOUS = [
    "catalog_danceability", "catalog_energy", "catalog_loudness",
    "catalog_speechiness", "catalog_acousticness", "catalog_instrumentalness",
    "catalog_liveness", "catalog_valence", "catalog_tempo", "catalog_release_year",
]
KEY_VALUES = list(range(12))
TIME_SIGNATURE_VALUES = list(range(6))

track_work = (
    joined_tracks
    .withColumn("duration_band_id",
        F.when(F.col("duration_seconds") < 120, 0)
        .when(F.col("duration_seconds") < 240, 1)
        .when(F.col("duration_seconds") < 360, 2).otherwise(3))
    .withColumn("duration_band",
        F.when(F.col("duration_band_id") == 0, "under_2_min")
        .when(F.col("duration_band_id") == 1, "2_to_4_min")
        .when(F.col("duration_band_id") == 2, "4_to_6_min").otherwise("over_6_min"))
    .withColumn("spud_popularity_band_id",
        F.when(F.col("spud_popularity") < 0.10, 0)
        .when(F.col("spud_popularity") < 0.30, 1)
        .when(F.col("spud_popularity") < 0.60, 2).otherwise(3))
    .withColumn("spud_popularity_band",
        F.when(F.col("spud_popularity_band_id") == 0, "low_under_0_10")
        .when(F.col("spud_popularity_band_id") == 1, "moderate_0_10_0_29")
        .when(F.col("spud_popularity_band_id") == 2, "popular_0_30_0_59").otherwise("high_0_60_plus"))
    .withColumn("release_decade",
        F.when(F.col("audio_features_available") == 1, (F.floor(F.col("catalog_release_year") / 10) * 10).cast("int")))
    .withColumn("release_decade_segment",
        F.coalesce(F.col("release_decade").cast("string"), F.lit("unknown_no_audio_match")))
    .withColumn("log_duration_seconds", F.log1p("duration_seconds"))
)

base_stats = track_work.agg(
    F.avg("log_duration_seconds").alias("duration_mean"),
    F.stddev_pop("log_duration_seconds").alias("duration_std"),
    F.avg("spud_popularity").alias("popularity_mean"),
    F.stddev_pop("spud_popularity").alias("popularity_std"),
).first().asDict()
track_work = (
    track_work
    .withColumn("z_log_duration_seconds",
        (F.col("log_duration_seconds") - base_stats["duration_mean"]) / base_stats["duration_std"])
    .withColumn("z_spud_popularity_context",
        (F.col("spud_popularity") - base_stats["popularity_mean"]) / base_stats["popularity_std"])
)

matched_audio_stats = track_work.where("audio_features_available = 1").agg(*[
    expression
    for column_name in AUDIO_CONTINUOUS
    for expression in (
        F.avg(column_name).alias(f"{column_name}__mean"),
        F.stddev_pop(column_name).alias(f"{column_name}__std"),
    )
]).first().asDict()
for column_name in AUDIO_CONTINUOUS:
    mean_value = float(matched_audio_stats[f"{column_name}__mean"])
    std_value = float(matched_audio_stats[f"{column_name}__std"] or 1.0)
    track_work = track_work.withColumn(
        f"z_{column_name}",
        F.when(F.col("audio_features_available") == 1, (F.col(column_name) - mean_value) / std_value).otherwise(0.0),
    )

track_title_tokenizer = RegexTokenizer(
    inputCol="track_title", outputCol="track_title_tokens",
    pattern=r"\W+", minTokenLength=2, toLowercase=True,
)
track_title_hasher = HashingTF(
    inputCol="track_title_tokens", outputCol="track_title_hash_vector",
    numFeatures=TRACK_TITLE_HASH_DIM, binary=True,
)
track_work = track_title_hasher.transform(track_title_tokenizer.transform(track_work)).withColumn(
    "track_title_hash_features", vector_to_array("track_title_hash_vector", "float64")
)

def one_hot(column_name, values):
    return F.array(*[
        F.when(F.col(column_name) == F.lit(value), 1.0).otherwise(0.0) for value in values
    ])

TRACK_AUDIO_Z_NAMES = [f"z_{name}" for name in AUDIO_CONTINUOUS]
TRACK_PREFIX_NAMES = (
    ["z_log_duration_seconds", "audio_features_available"] + TRACK_AUDIO_Z_NAMES
    + ["catalog_explicit", "catalog_mode", "catalog_tempo_was_missing"]
    + [f"key_{value}" for value in KEY_VALUES]
    + [f"time_signature_{value}" for value in TIME_SIGNATURE_VALUES]
)
TRACK_TITLE_FEATURE_NAMES = [f"title_hash_{index:02d}" for index in range(TRACK_TITLE_HASH_DIM)]
TRACK_SAFE_FEATURE_NAMES = TRACK_PREFIX_NAMES + TRACK_TITLE_FEATURE_NAMES
TRACK_SAFE_NO_TIME_FEATURE_NAMES = [name for name in TRACK_SAFE_FEATURE_NAMES if name != "z_catalog_release_year"]
TRACK_CONTEXT_FEATURE_NAMES = TRACK_SAFE_FEATURE_NAMES + ["z_spud_popularity_context"]

track_prefix_array = F.concat(
    F.array(*[F.col(name).cast("double") for name in ["z_log_duration_seconds", "audio_features_available"] + TRACK_AUDIO_Z_NAMES]),
    F.array(
        F.coalesce(F.col("catalog_explicit").cast("double"), F.lit(0.0)),
        F.coalesce(F.col("catalog_mode").cast("double"), F.lit(0.0)),
        F.coalesce(F.col("catalog_tempo_was_missing").cast("double"), F.lit(0.0)),
    ),
    one_hot("catalog_key", KEY_VALUES),
    one_hot("catalog_time_signature", TIME_SIGNATURE_VALUES),
)
track_work = (
    track_work
    .withColumn("features_safe", F.concat(track_prefix_array, F.col("track_title_hash_features")))
    .withColumn("features_safe_no_time", F.concat(
        F.array(*[
            F.col(name).cast("double")
            for name in ["z_log_duration_seconds", "audio_features_available"] + TRACK_AUDIO_Z_NAMES
            if name != "z_catalog_release_year"
        ]),
        F.array(
            F.coalesce(F.col("catalog_explicit").cast("double"), F.lit(0.0)),
            F.coalesce(F.col("catalog_mode").cast("double"), F.lit(0.0)),
            F.coalesce(F.col("catalog_tempo_was_missing").cast("double"), F.lit(0.0)),
        ),
        one_hot("catalog_key", KEY_VALUES), one_hot("catalog_time_signature", TIME_SIGNATURE_VALUES),
        F.col("track_title_hash_features"),
    ))
    .withColumn("features_with_popularity_context", F.concat(
        F.col("features_safe"), F.array(F.col("z_spud_popularity_context").cast("double"))
    ))
)

track_window = Window.orderBy("spotify_track_id")
track_nodes = (
    track_work
    .withColumn("track_node_id", (F.row_number().over(track_window) - 1).cast("long"))
    .select(
        "track_node_id", "spotify_track_id", "spud_track_id", "track_title",
        "spotify_artist_id", "artist_name", "spud_popularity",
        "audio_features_available", "duration_band_id", "duration_band",
        "spud_popularity_band_id", "spud_popularity_band",
        "release_decade", "release_decade_segment",
        "features_safe", "features_safe_no_time", "features_with_popularity_context",
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
track_node_count = track_nodes.count()
print(f"Track feature dimensions: safe={len(TRACK_SAFE_FEATURE_NAMES)}, no_time={len(TRACK_SAFE_NO_TIME_FEATURE_NAMES)}, context={len(TRACK_CONTEXT_FEATURE_NAMES)}")

## 5. Encode connected playlist nodes

Only playlists with at least one edge become graph nodes. Title tokens are hashed into a fixed vector. Full-graph size and duration are retained as descriptive fields but deliberately excluded from the safe vector because they include relationships that will later be hidden for evaluation.

In [ ]:
PLAYLIST_TITLE_HASH_DIM = 64
connected_playlists = (
    playlist_dimension.where(F.col("track_count") > 0)
    .join(playlist_audio_coverage.select("playlist_id", "audio_matched_track_count", "audio_coverage_ratio"), "playlist_id", "left")
)
playlist_tokenizer = RegexTokenizer(
    inputCol="playlist_title", outputCol="playlist_title_tokens",
    pattern=r"\W+", minTokenLength=2, toLowercase=True,
)
playlist_hasher = HashingTF(
    inputCol="playlist_title_tokens", outputCol="playlist_title_hash_vector",
    numFeatures=PLAYLIST_TITLE_HASH_DIM, binary=True,
)
playlist_work = playlist_hasher.transform(playlist_tokenizer.transform(connected_playlists)).withColumn(
    "playlist_title_hash_features", vector_to_array("playlist_title_hash_vector", "float64")
)
PLAYLIST_FEATURE_NAMES = ["title_was_missing"] + [
    f"title_hash_{index:02d}" for index in range(PLAYLIST_TITLE_HASH_DIM)
]
playlist_work = playlist_work.withColumn(
    "features_safe", F.concat(
        F.array(F.col("title_was_missing").cast("double")),
        F.col("playlist_title_hash_features"),
    ),
)
playlist_window = Window.orderBy("playlist_id")
playlist_nodes = (
    playlist_work
    .withColumn("playlist_node_id", (F.row_number().over(playlist_window) - 1).cast("long"))
    .select(
        "playlist_node_id", "playlist_id", "playlist_title",
        "track_count", "artist_count", "computed_duration_seconds",
        "playlist_size_band_id", "playlist_size_band", "eligible_for_link_prediction",
        "audio_matched_track_count", "audio_coverage_ratio", "features_safe",
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
playlist_node_count = playlist_nodes.count()
print(f"Playlist nodes: {playlist_node_count:,}; safe feature dimension: {len(PLAYLIST_FEATURE_NAMES)}")

## 6. Join and encode connected artist nodes

SPUD supplies an artist identity for every track. Followers and genres are joined from the original artist catalog where available; missing catalog metadata receives a zero standardized value and explicit missingness flags. Artist popularity remains optional context.

In [ ]:
current_artist_dimension = (
    catalog_artist_relationships.groupBy("artist_id").agg(
        F.first("artist_name", ignorenulls=True).alias("catalog_artist_name"),
        F.max("artist_followers").alias("artist_followers"),
        F.max("artist_popularity").alias("artist_popularity_context"),
        F.first("artist_genres", ignorenulls=True).alias("artist_genres_raw"),
        F.max(F.col("artist_metadata_found").cast("int")).alias("artist_metadata_available"),
    )
    .withColumnRenamed("artist_id", "spotify_artist_id")
)
spud_artist_dimension = connected_spud_tracks.groupBy("spotify_artist_id").agg(
    F.first("artist_name", ignorenulls=True).alias("artist_name")
)
artist_work = (
    spud_artist_dimension.join(current_artist_dimension, "spotify_artist_id", "left")
    .withColumn("artist_metadata_available", F.coalesce("artist_metadata_available", F.lit(0)))
    .withColumn("followers_missing", F.col("artist_followers").isNull().cast("int"))
    .withColumn("popularity_context_missing", F.col("artist_popularity_context").isNull().cast("int"))
    .withColumn("genres_missing",
        (F.col("artist_genres_raw").isNull() | (F.trim("artist_genres_raw") == "[]")).cast("int"))
)
strip_list_punctuation = "[]'"
artist_work = artist_work.withColumn(
    "genres",
    F.when(F.col("genres_missing") == 1, F.array().cast("array<string>")).otherwise(
        F.filter(
            F.transform(
                F.split(F.translate("artist_genres_raw", strip_list_punctuation, ""), r",\s*"),
                lambda genre: F.lower(F.trim(genre)),
            ),
            lambda genre: F.length(genre) > 0,
        )
    ),
)
artist_numeric_stats = artist_work.where(F.col("artist_followers").isNotNull()).select(
    F.log1p("artist_followers").alias("log_followers"),
    F.col("artist_popularity_context").cast("double").alias("artist_popularity_context"),
).agg(
    F.avg("log_followers").alias("followers_mean"),
    F.stddev_pop("log_followers").alias("followers_std"),
    F.avg("artist_popularity_context").alias("popularity_mean"),
    F.stddev_pop("artist_popularity_context").alias("popularity_std"),
).first().asDict()
artist_work = (
    artist_work
    .withColumn("z_log_followers",
        F.when(F.col("artist_followers").isNotNull(),
            (F.log1p("artist_followers") - artist_numeric_stats["followers_mean"]) / artist_numeric_stats["followers_std"])
        .otherwise(0.0))
    .withColumn("z_artist_popularity_context",
        F.when(F.col("artist_popularity_context").isNotNull(),
            (F.col("artist_popularity_context") - artist_numeric_stats["popularity_mean"]) / artist_numeric_stats["popularity_std"])
        .otherwise(0.0))
    .withColumn("follower_band_id",
        F.when(F.col("followers_missing") == 1, 0)
        .when(F.col("artist_followers") < 1000, 1)
        .when(F.col("artist_followers") < 10000, 2)
        .when(F.col("artist_followers") < 100000, 3)
        .when(F.col("artist_followers") < 1000000, 4).otherwise(5))
    .withColumn("follower_band",
        F.when(F.col("follower_band_id") == 0, "unknown")
        .when(F.col("follower_band_id") == 1, "under_1k")
        .when(F.col("follower_band_id") == 2, "1k_10k")
        .when(F.col("follower_band_id") == 3, "10k_100k")
        .when(F.col("follower_band_id") == 4, "100k_1m").otherwise("1m_plus"))
)

ARTIST_GENRE_HASH_DIM = 64
artist_genre_hasher = HashingTF(
    inputCol="genres", outputCol="artist_genre_hash_vector",
    numFeatures=ARTIST_GENRE_HASH_DIM, binary=True,
)
artist_work = artist_genre_hasher.transform(artist_work).withColumn(
    "artist_genre_hash_features", vector_to_array("artist_genre_hash_vector", "float64")
)
ARTIST_SAFE_PREFIX_NAMES = [
    "artist_metadata_available", "followers_missing", "genres_missing", "z_log_followers",
]
ARTIST_GENRE_FEATURE_NAMES = [f"genre_hash_{index:02d}" for index in range(ARTIST_GENRE_HASH_DIM)]
ARTIST_SAFE_FEATURE_NAMES = ARTIST_SAFE_PREFIX_NAMES + ARTIST_GENRE_FEATURE_NAMES
ARTIST_CONTEXT_FEATURE_NAMES = ARTIST_SAFE_FEATURE_NAMES + [
    "z_artist_popularity_context", "popularity_context_missing",
]
artist_work = (
    artist_work
    .withColumn("features_safe", F.concat(
        F.array(*[F.col(name).cast("double") for name in ARTIST_SAFE_PREFIX_NAMES]),
        F.col("artist_genre_hash_features"),
    ))
    .withColumn("features_with_popularity_context", F.concat(
        F.col("features_safe"),
        F.array(
            F.col("z_artist_popularity_context").cast("double"),
            F.col("popularity_context_missing").cast("double"),
        ),
    ))
)
artist_window = Window.orderBy("spotify_artist_id")
artist_nodes = (
    artist_work
    .withColumn("artist_node_id", (F.row_number().over(artist_window) - 1).cast("long"))
    .select(
        "artist_node_id", "spotify_artist_id", "artist_name",
        "artist_metadata_available", "artist_followers", "follower_band_id", "follower_band",
        "artist_popularity_context", "genres", "features_safe", "features_with_popularity_context",
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
artist_node_count = artist_nodes.count()
artist_metadata_match_count = artist_nodes.where("artist_metadata_available = 1").count()
print(f"Artist nodes: {artist_node_count:,}; catalog metadata matches: {artist_metadata_match_count:,}")
print(f"Artist feature dimensions: safe={len(ARTIST_SAFE_FEATURE_NAMES)}, context={len(ARTIST_CONTEXT_FEATURE_NAMES)}")

## 7. Create the two edge types

The playlist-track relation is the link-prediction target. SPUD has no track order or edge timestamp, so there is intentionally no `position` column. Full-graph degrees are saved only for descriptive analysis and must be recomputed from training edges after the split.

In [ ]:
playlist_id_map = playlist_nodes.select("playlist_id", "playlist_node_id")
track_id_map = track_nodes.select("spotify_track_id", "track_node_id")
artist_id_map = artist_nodes.select("spotify_artist_id", "artist_node_id")

playlist_track_edges = (
    playlist_edge_source
    .join(playlist_id_map, on="playlist_id", how="inner")
    .join(track_id_map, on="spotify_track_id", how="inner")
    .groupBy("playlist_node_id", "track_node_id", "playlist_id", "spotify_track_id").agg(
        F.first("spud_track_id").alias("spud_track_id")
    )
)
playlist_degrees = playlist_track_edges.groupBy("playlist_node_id").count().withColumnRenamed("count", "full_playlist_degree")
track_playlist_degrees = playlist_track_edges.groupBy("track_node_id").count().withColumnRenamed("count", "full_track_playlist_degree")
playlist_track_edges = (
    playlist_track_edges.join(playlist_degrees, "playlist_node_id", "inner")
    .join(track_playlist_degrees, "track_node_id", "inner")
    .select(
        F.col("playlist_node_id").alias("src_playlist_node_id"),
        F.col("track_node_id").alias("dst_track_node_id"),
        "playlist_id", "spotify_track_id", "spud_track_id",
        "full_playlist_degree", "full_track_playlist_degree",
    )
    .persist(StorageLevel.MEMORY_AND_DISK)
)
playlist_track_edge_count = playlist_track_edges.count()

track_artist_edges = (
    track_nodes.select("track_node_id", "spotify_track_id", "spotify_artist_id")
    .join(artist_id_map, on="spotify_artist_id", how="inner")
    .select(
        F.col("track_node_id").alias("src_track_node_id"),
        F.col("artist_node_id").alias("dst_artist_node_id"),
        "spotify_track_id", "spotify_artist_id",
    )
    .dropDuplicates(["src_track_node_id", "dst_artist_node_id"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
track_artist_edge_count = track_artist_edges.count()
print(f"Playlist-track target edges: {playlist_track_edge_count:,}")
print(f"Track-artist context edges : {track_artist_edge_count:,}")

## 8. Segment distributions and join-coverage observations

In [ ]:
def distribution(dataframe, column_name, total_count, entity):
    return (
        dataframe.groupBy(column_name).count()
        .withColumn("percentage", F.round(100 * F.col("count") / F.lit(total_count), 3))
        .withColumn("entity", F.lit(entity))
        .withColumn("segment", F.lit(column_name))
        .select("entity", "segment", F.col(column_name).cast("string").alias("value"), "count", "percentage")
    )

segment_specs = [
    (playlist_nodes, "playlist_size_band", playlist_node_count, "playlist"),
    (playlist_nodes, "eligible_for_link_prediction", playlist_node_count, "playlist"),
    (track_nodes, "duration_band", track_node_count, "track"),
    (track_nodes, "spud_popularity_band", track_node_count, "track"),
    (track_nodes, "audio_features_available", track_node_count, "track"),
    (track_nodes, "release_decade_segment", track_node_count, "track"),
    (artist_nodes, "follower_band", artist_node_count, "artist"),
    (artist_nodes, "artist_metadata_available", artist_node_count, "artist"),
]
segment_distribution = distribution(*segment_specs[0])
for spec in segment_specs[1:]:
    segment_distribution = segment_distribution.unionByName(distribution(*spec))
segment_distribution_pdf = segment_distribution.orderBy("entity", "segment", F.desc("count")).toPandas()
display(segment_distribution_pdf)

## 9. Validate graph integrity and fixed feature arrays

In [ ]:
def invalid_array_count(dataframe, column_name):
    return dataframe.where(F.exists(
        F.col(column_name),
        lambda value: value.isNull() | F.isnan(value) | (F.abs(value) > F.lit(1.0e308)),
    )).count()

def contiguous_index_audit(dataframe, id_column):
    return dataframe.agg(
        F.min(id_column).alias("min_id"), F.max(id_column).alias("max_id"),
        F.countDistinct(id_column).alias("distinct_ids"),
    ).first().asDict()

validation_metrics = {
    "duplicate_playlist_ids": playlist_nodes.groupBy("playlist_id").count().where("count > 1").count(),
    "duplicate_track_ids": track_nodes.groupBy("spotify_track_id").count().where("count > 1").count(),
    "duplicate_artist_ids": artist_nodes.groupBy("spotify_artist_id").count().where("count > 1").count(),
    "duplicate_playlist_track_edges": playlist_track_edges.groupBy("src_playlist_node_id", "dst_track_node_id").count().where("count > 1").count(),
    "duplicate_track_artist_edges": track_artist_edges.groupBy("src_track_node_id", "dst_artist_node_id").count().where("count > 1").count(),
    "invalid_playlist_features": invalid_array_count(playlist_nodes, "features_safe"),
    "invalid_track_features": invalid_array_count(track_nodes, "features_safe"),
    "invalid_artist_features": invalid_array_count(artist_nodes, "features_safe"),
    "playlist_feature_size_errors": playlist_nodes.where(F.size("features_safe") != len(PLAYLIST_FEATURE_NAMES)).count(),
    "track_feature_size_errors": track_nodes.where(F.size("features_safe") != len(TRACK_SAFE_FEATURE_NAMES)).count(),
    "artist_feature_size_errors": artist_nodes.where(F.size("features_safe") != len(ARTIST_SAFE_FEATURE_NAMES)).count(),
    "missing_playlist_endpoints": playlist_track_edges.join(
        playlist_nodes.select(F.col("playlist_node_id").alias("src_playlist_node_id")), "src_playlist_node_id", "left_anti"
    ).count(),
    "missing_playlist_track_endpoints": playlist_track_edges.join(
        track_nodes.select(F.col("track_node_id").alias("dst_track_node_id")), "dst_track_node_id", "left_anti"
    ).count(),
    "missing_context_track_endpoints": track_artist_edges.join(
        track_nodes.select(F.col("track_node_id").alias("src_track_node_id")), "src_track_node_id", "left_anti"
    ).count(),
    "missing_artist_endpoints": track_artist_edges.join(
        artist_nodes.select(F.col("artist_node_id").alias("dst_artist_node_id")), "dst_artist_node_id", "left_anti"
    ).count(),
}
playlist_index_audit = contiguous_index_audit(playlist_nodes, "playlist_node_id")
track_index_audit = contiguous_index_audit(track_nodes, "track_node_id")
artist_index_audit = contiguous_index_audit(artist_nodes, "artist_node_id")
assert playlist_index_audit == {"min_id": 0, "max_id": playlist_node_count - 1, "distinct_ids": playlist_node_count}
assert track_index_audit == {"min_id": 0, "max_id": track_node_count - 1, "distinct_ids": track_node_count}
assert artist_index_audit == {"min_id": 0, "max_id": artist_node_count - 1, "distinct_ids": artist_node_count}
assert playlist_track_edge_count == input_counts["spud_playlist_track_edges"]
assert track_artist_edge_count == track_node_count
assert all(value == 0 for value in validation_metrics.values())
print("INTEGRATED GRAPH VALIDATION PASSED")
display(pd.DataFrame(validation_metrics.items(), columns=["metric", "affected_rows"]))

## 10. Save graph tables, feature catalog, and human-readable reports

In [ ]:
feature_catalog_rows = []
for node_type, vector_name, names in [
    ("playlist", "features_safe", PLAYLIST_FEATURE_NAMES),
    ("track", "features_with_popularity_context", TRACK_CONTEXT_FEATURE_NAMES),
    ("artist", "features_with_popularity_context", ARTIST_CONTEXT_FEATURE_NAMES),
]:
    for index, feature_name in enumerate(names):
        safe = not ("popularity_context" in feature_name)
        if node_type == "track" and feature_name == "z_catalog_release_year":
            note = "safe with cohort caution; also supplied in a no-time vector"
        elif safe:
            note = "included in safe feature vector"
        else:
            note = "optional context; exclude from primary leakage-safe experiment"
        feature_catalog_rows.append((node_type, vector_name, index, feature_name, safe, note))
feature_catalog = spark.createDataFrame(
    feature_catalog_rows,
    ["node_type", "vector_column", "feature_index", "feature_name", "safe_primary_input", "modeling_note"],
)

(playlist_nodes.repartition(4).write.mode("overwrite").option("compression", "snappy").parquet(str(PLAYLIST_NODES_PATH)))
(track_nodes.repartition(8).write.mode("overwrite").option("compression", "snappy").parquet(str(TRACK_NODES_PATH)))
(artist_nodes.repartition(8).write.mode("overwrite").option("compression", "snappy").parquet(str(ARTIST_NODES_PATH)))
(playlist_track_edges.repartition(8).write.mode("overwrite").option("compression", "snappy").parquet(str(PLAYLIST_TRACK_EDGES_PATH)))
(track_artist_edges.repartition(8).write.mode("overwrite").option("compression", "snappy").parquet(str(TRACK_ARTIST_EDGES_PATH)))
(feature_catalog.coalesce(1).write.mode("overwrite").parquet(str(FEATURE_CATALOG_PATH)))

feature_catalog.orderBy("node_type", "feature_index").toPandas().to_csv(
    REPORTS_ROOT / "integrated_gnn_feature_catalog.csv", index=False
)
segment_distribution_pdf.to_csv(REPORTS_ROOT / "integrated_graph_segments.csv", index=False)
join_coverage_report = {
    "connected_playlists": playlist_node_count,
    "connected_tracks": track_node_count,
    "connected_artists": artist_node_count,
    "playlist_track_edges": playlist_track_edge_count,
    "track_artist_edges": track_artist_edge_count,
    "tracks_with_catalog_audio": audio_match_count,
    "catalog_audio_coverage_percent": audio_match_pct,
    "artists_with_catalog_metadata": artist_metadata_match_count,
    "edge_order_available": False,
}
(REPORTS_ROOT / "integrated_graph_coverage.json").write_text(
    json.dumps(join_coverage_report, indent=2), encoding="utf-8"
)
print("Saved integrated graph outputs under data/processed/gnn_integrated")

## 11. Read back outputs and print every final data type

In [ ]:
saved_outputs = {
    "playlist_nodes": spark.read.parquet(str(PLAYLIST_NODES_PATH)),
    "track_nodes": spark.read.parquet(str(TRACK_NODES_PATH)),
    "artist_nodes": spark.read.parquet(str(ARTIST_NODES_PATH)),
    "playlist_track_edges": spark.read.parquet(str(PLAYLIST_TRACK_EDGES_PATH)),
    "track_artist_edges": spark.read.parquet(str(TRACK_ARTIST_EDGES_PATH)),
    "feature_catalog": spark.read.parquet(str(FEATURE_CATALOG_PATH)),
}
saved_counts = {name: dataframe.count() for name, dataframe in saved_outputs.items()}
assert saved_counts["playlist_nodes"] == playlist_node_count
assert saved_counts["track_nodes"] == track_node_count
assert saved_counts["artist_nodes"] == artist_node_count
assert saved_counts["playlist_track_edges"] == playlist_track_edge_count
assert saved_counts["track_artist_edges"] == track_artist_edge_count
output_schema_pdf = pd.concat([
    schema_as_pandas(name, dataframe) for name, dataframe in saved_outputs.items()
], ignore_index=True)
display(output_schema_pdf)
display(pd.DataFrame(saved_counts.items(), columns=["dataset", "rows"]))

## 12. Computed observations and modeling hand-off

In [ ]:
eligible_graph_playlists = playlist_nodes.where("eligible_for_link_prediction = 1").count()
playlists_with_five_audio_matches = playlist_audio_coverage.where("audio_matched_track_count >= 5").count()
artist_metadata_pct = 100 * artist_metadata_match_count / artist_node_count

print("OBSERVATIONS")
print("============")
print(f"1. The integrated graph has {playlist_node_count:,} playlists, {track_node_count:,} tracks, and {artist_node_count:,} artists.")
print(f"2. Its target relation contains {playlist_track_edge_count:,} playlist-track links; the context relation contains {track_artist_edge_count:,} track-artist links.")
print(f"3. {eligible_graph_playlists:,} connected playlists have at least five tracks and can support seed/hidden-track evaluation.")
print(f"4. Only {audio_match_count:,} playlist tracks ({audio_match_pct:.3f}%) match the original 600k-track audio catalog.")
print("5. All unmatched playlist tracks were retained; they can learn collaborative ID embeddings from playlist membership.")
print(f"6. {playlists_with_five_audio_matches:,} playlists have at least five audio-matched tracks, so an audio-only restricted experiment would be much smaller and biased.")
print(f"7. Original follower/genre metadata is available for {artist_metadata_match_count:,} artists ({artist_metadata_pct:.3f}%); missingness is encoded explicitly.")
print(f"8. Safe feature sizes are playlist={len(PLAYLIST_FEATURE_NAMES)}, track={len(TRACK_SAFE_FEATURE_NAMES)}, artist={len(ARTIST_SAFE_FEATURE_NAMES)}.")
print("9. Popularity fields are excluded from primary safe vectors and available only in optional context vectors for ablation testing.")
print("10. Full-graph degrees are descriptive only; the model notebook must recalculate degree normalization using training edges after hiding validation/test links.")
print("11. Because SPUD provides neither track order nor edge timestamps, the valid task is set-based playlist completion, not next-song sequence prediction.")
print("12. The next notebook should split positive edges per eligible playlist, sample negatives after splitting, and evaluate Recall@K, NDCG@K, HitRate@K, MRR, and coverage.")